this notebook contains cleaning for the incident isochrone path.

# 1.0 Load packages and data

In [ ]:
# load the packages
import pandas as pd
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from shapely.wkt import loads
from sklearn.neighbors import NearestNeighbors
from scipy.stats import chi2_contingency

In [ ]:
# load the crime data
incident_gdf = gpd.read_file('incident_gdf.geojson', use_arrow=True)

In [ ]:
isochrone_df = gpd.read_file('school_catchments.geojson')

In [ ]:
# Load the previous cleaned bradford 
bradford_boundary = gpd.read_file('bradford_boundary.geojson')

There is no deep data cleaning required here. The datasets have been cleaned from previous buffer approaches.

# 2.0 Data Previewing

## 2.1 Incident Data

In [ ]:
# view crime data
incident_gdf.head(3)

In [ ]:
# check the shape of the data
incident_gdf.shape

In [ ]:
# check data 
incident_gdf.info()

In [ ]:
# confirm null values
incident_gdf.isnull().sum()

In [ ]:
print(incident_gdf.crs)

In [ ]:
# reproject the cordinates
incident_gdf = incident_gdf.to_crs("EPSG:4326")

## 2.2 Boundary

In [ ]:
print(bradford_boundary.crs)

In [ ]:
# convert to the angle
bradford_boundary = bradford_boundary.to_crs("EPSG:4326")

## 2.3 School Data

In [ ]:
# check data structure
isochrone_df.shape

In [ ]:
# check teh datatypes
isochrone_df.info()

In [ ]:
# confirm column names
isochrone_df.columns

# 3.0 Data cleaning

## 3.1 Isochrone data

In [ ]:
# check special schools
special_schools = isochrone_df[isochrone_df['school_name'].str.contains('Special', case=False, na=False)]

In [ ]:
# remove the special schools
isochrone_df = isochrone_df.drop(special_schools.index, axis=0)

In [ ]:
# drop unneeded columns
isochrone_df = isochrone_df.drop('osm_id', axis=1)

In [ ]:
isochrone_df = gpd.GeoDataFrame(isochrone_df, geometry='geometry', crs="EPSG:4326")

In [ ]:
print(isochrone_df.crs)

# 4.0 Data selection

I had used three speeds for data extraction, building on pinna & Murau 2018 which found that the speed of children aged 0-18 is 1.0m/s +/- 0.11. I will only apply with the mean speed for this; further work could identify the differences.

In [ ]:
isochrone_df = isochrone_df[isochrone_df['walk_speed_ms'] == 1.04]

Merge the datasets

In [ ]:
# join the crime data and the isochrone
incident_isochrones = gpd.sjoin(
    incident_gdf, 
    isochrone_df, 
    how='left', 
    predicate='within'
).reset_index()

In [ ]:
# sort by travel time and take first record for each crime
incident_isochrones = incident_isochrones.sort_values(by='trip_time_min', ascending=True)

# drop duplicates and keep one
incident_isochrones = incident_isochrones.drop_duplicates(subset='index', keep='first')

# set index
incident_isochrones = incident_isochrones.set_index('index')

In [ ]:
incident_isochrones.shape

### Save data 

In [ ]:
# save all the data
df = incident_isochrones.copy()
    
df.to_parquet('incident_isochrones.parquet', index=False)
df.to_file('incident_isochrones.geojson', driver='GeoJSON', index=False)